In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import os
import glob
from sklearn.tree import DecisionTreeClassifier, export_text
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None
display(HTML("<style>.container { width:100% !important; }</style>"))

DATA_DIR = r'D:\0dot1_Aug_2016_master\data\mstock_mtf_daily_data'


In [2]:
# 2. Data Loading & Splits
def load_all_data():
    all_data = []
    files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
    import random
    random.seed(42)
    random.shuffle(files)
    
    for file_path in files:
        if len(all_data) >= 100:
            break
        symbol = os.path.basename(file_path).replace('.csv', '')
        try:
            df = pd.read_csv(file_path)
            df = df.dropna(subset=['Close']) # Data Cleaning: Remove empty rows
            
            # Price Filter: Between 90 and 600 Rs
            if len(df) > 0:
                latest_price = df['Close'].iloc[-1]
                if latest_price < 90 or latest_price > 600:
                    continue
                    
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.sort_values('Date').reset_index(drop=True)
            
            # Anomaly Handling: Remove extreme gaps/splits (>20% move)
            df['Daily_Ret'] = df['Close'].pct_change()
            df = df[(df['Daily_Ret'].isna()) | (df['Daily_Ret'].abs() <= 0.20)]
            
            df['Symbol'] = symbol
            all_data.append(df)
        except Exception as e:
            pass
            
    return pd.concat(all_data, ignore_index=True)

data = load_all_data()

# Split Stocks into Train/Validate (First 50) and Test (Last 50) randomly
unique_symbols = list(data['Symbol'].unique())
import random
random.seed(42)
random.shuffle(unique_symbols)
mid = len(unique_symbols) // 2
train_symbols = unique_symbols[:mid]
test_symbols = unique_symbols[mid:]

train_data_raw = data[data['Symbol'].isin(train_symbols)]
test_data_raw = data[data['Symbol'].isin(test_symbols)]

print(f"Total Stocks: {len(unique_symbols)} | Train Set: {len(train_symbols)} | Test Set: {len(test_symbols)}")


Total Stocks: 100 | Train Set: 50 | Test Set: 50


In [3]:
# 3. Feature Engineering
LOOKBACKS = [13, 21, 34]
HOLD_DAYS = 13

def enrich_data(df):
    df = df.copy()
    
    # Target Variable (13-day Forward Return)
    df['Fwd_Ret'] = df['Close'].shift(-HOLD_DAYS) / df['Close'] - 1
    
    # Features
    for p in LOOKBACKS:
        # ER
        change = abs(df['Close'] - df['Close'].shift(p))
        volatility = df['Close'].diff().abs().rolling(p).sum()
        df[f'ER_{p}'] = change / volatility
        
        # WMA Distance
        weights = np.arange(1, p + 1)
        wma = df['Close'].rolling(p).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
        df[f'WMA_Dist_{p}'] = (df['Close'] - wma) / wma
        
    return df

print("Calculating features across the dataset...")
_train_list = []
for sym, grp in train_data_raw.groupby('Symbol'):
    _train_list.append(enrich_data(grp))
train_data = pd.concat(_train_list, ignore_index=True).dropna()

_test_list = []
for sym, grp in test_data_raw.groupby('Symbol'):
    _test_list.append(enrich_data(grp))
test_data = pd.concat(_test_list, ignore_index=True).dropna()
print("Feature Engineering Complete.")


Calculating features across the dataset...


Feature Engineering Complete.


In [4]:
# 4. Phase 1: Train Base Entry Rules (Pre-2020 Data)
phase_1_data = train_data[train_data['Date'] < '2020-01-01']

# Target: Did it generate a positive return?
phase_1_data['Target_Win'] = (phase_1_data['Fwd_Ret'] > 0).astype(int)

# Features: The Efficiency Ratios
entry_features = [f'ER_{p}' for p in LOOKBACKS]
X_entry = phase_1_data[entry_features]
y_entry = phase_1_data['Target_Win']

# Train Decision Tree (Max Depth 1 for a single crisp rule)
clf_entry = DecisionTreeClassifier(max_depth=1, random_state=42, class_weight='balanced')
clf_entry.fit(X_entry, y_entry)

# Extract Rule
tree_rules = export_text(clf_entry, feature_names=entry_features)
print("=== PHASE 1: Base Entry Decision Tree ===")
print(tree_rules)

# Parse the root rule manually for the pipeline
feature_idx = clf_entry.tree_.feature[0]
entry_feature_name = entry_features[feature_idx]
entry_threshold = clf_entry.tree_.threshold[0]

# Determine which side of the split predicts a Win (Class 1)
left_pred = np.argmax(clf_entry.tree_.value[1])
right_pred = np.argmax(clf_entry.tree_.value[2])

if right_pred == 1:
    print(f"🔥 CHAMPION BASE RULE: Enter Long when {entry_feature_name} > {entry_threshold:.3f}")
    def apply_base_rule(df): return df[entry_feature_name] > entry_threshold
else:
    print(f"🔥 CHAMPION BASE RULE: Enter Long when {entry_feature_name} <= {entry_threshold:.3f}")
    def apply_base_rule(df): return df[entry_feature_name] <= entry_threshold


=== PHASE 1: Base Entry Decision Tree ===
|--- ER_34 <= 0.78
|   |--- class: 0
|--- ER_34 >  0.78
|   |--- class: 1

🔥 CHAMPION BASE RULE: Enter Long when ER_34 > 0.782


In [5]:
# 5. Phase 2: Learn Ignore Rules (2020 to 2022 Data)
phase_2_data = train_data[(train_data['Date'] >= '2020-01-01') & (train_data['Date'] < '2023-01-01')]

# Filter down to ONLY trades where the Base Rule fired
phase_2_triggered = phase_2_data[apply_base_rule(phase_2_data)]

# Target: Did it generate a LOSS? (We are learning what predicts failure)
phase_2_triggered['Target_Loss'] = (phase_2_triggered['Fwd_Ret'] < 0).astype(int)

# Features: The Avoidance / Risk Features
ignore_features = [f'WMA_Dist_{p}' for p in LOOKBACKS]
X_ignore = phase_2_triggered[ignore_features]
y_ignore = phase_2_triggered['Target_Loss']

if len(X_ignore) < 50:
    print("Not enough triggered trades in Phase 2 to learn ignore rules securely.")
else:
    # Train Decision Tree (Max Depth 1)
    clf_ignore = DecisionTreeClassifier(max_depth=1, random_state=42, class_weight='balanced')
    clf_ignore.fit(X_ignore, y_ignore)
    
    print("\n=== PHASE 2: Risk Avoidance Decision Tree ===")
    print(export_text(clf_ignore, feature_names=ignore_features))
    
    feature_idx = clf_ignore.tree_.feature[0]
    ignore_feature_name = ignore_features[feature_idx]
    ignore_threshold = clf_ignore.tree_.threshold[0]
    
    left_pred = np.argmax(clf_ignore.tree_.value[1])
    right_pred = np.argmax(clf_ignore.tree_.value[2])
    
    if right_pred == 1: # Right side predicts a LOSS
        print(f"🛡️ CHAMPION IGNORE RULE: Avoid trade if {ignore_feature_name} > {ignore_threshold:.3f}")
        def apply_ignore_rule(df): return df[ignore_feature_name] > ignore_threshold
    else: # Left side predicts a LOSS
        print(f"🛡️ CHAMPION IGNORE RULE: Avoid trade if {ignore_feature_name} <= {ignore_threshold:.3f}")
        def apply_ignore_rule(df): return df[ignore_feature_name] <= ignore_threshold


Not enough triggered trades in Phase 2 to learn ignore rules securely.


In [6]:
# 6. Phase 3: Unseen Backtest (2023+ Data on Unseen Stocks)
phase_3_data = test_data[test_data['Date'] >= '2023-01-01']

print("\n=== PHASE 3: Unseen Backtest ===")
print("Testing the ML Pipeline on strictly out-of-sample Timeframes AND out-of-sample Stocks!")

# Apply Base Rule
base_trades = phase_3_data[apply_base_rule(phase_3_data)]
print(f"Base ML Rule triggered {len(base_trades)} times.")

if len(base_trades) > 0:
    base_winrate = (base_trades['Fwd_Ret'] > 0).mean() * 100
    base_avg_ret = base_trades['Fwd_Ret'].mean() * 100
    print(f"Performance without Ignore Rule: Win Rate = {base_winrate:.2f}% | Avg Return = {base_avg_ret:.2f}%")

# Apply Ignore Rule Filter
if 'apply_ignore_rule' in globals():
    final_trades = base_trades[~apply_ignore_rule(base_trades)]
    print(f"After ML Ignore Filter, {len(final_trades)} trades remain.")
    
    if len(final_trades) > 0:
        final_winrate = (final_trades['Fwd_Ret'] > 0).mean() * 100
        final_avg_ret = final_trades['Fwd_Ret'].mean() * 100
        
        display(HTML(f"<h2>Final ML Pipeline Results</h2>"))
        display(HTML(f"<b>Trades Executed:</b> {len(final_trades)}<br>"))
        display(HTML(f"<b>Win Rate:</b> {final_winrate:.2f}%<br>"))
        display(HTML(f"<b>Average Return per Trade (13 Days):</b> +{final_avg_ret:.2f}%<br>"))
        
        print("\nDetailed Sample of Final Trades:")
        display(final_trades[['Date', 'Symbol', 'Close', 'Fwd_Ret']].head(15))
    else:
        print("No trades passed both ML filters in Phase 3.")



=== PHASE 3: Unseen Backtest ===
Testing the ML Pipeline on strictly out-of-sample Timeframes AND out-of-sample Stocks!
Base ML Rule triggered 9 times.
Performance without Ignore Rule: Win Rate = 0.00% | Avg Return = -15.64%
